# Table 3: comparison of downward-continuation approaches on field data

Run notebooks `01`–`10` first (or `scripts/run_all.sh`). Each writes its rows to `../results/*.json`.
This notebook collects them into Table 3, checks every row against the reference values in
`gravdc/table3.py`, and writes `../results/table3.csv` and `../results/table3.md`.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import csv
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from gravdc.table3 import METRICS, ORDER, TABLE3, matches

RESULTS_DIR = "../results"
runs = {}
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith(".json"):
        r = json.load(open(os.path.join(RESULTS_DIR, f)))
        runs[r["method"]] = r
missing = [n for n in ORDER if n not in runs]
assert not missing, f"run notebooks 01-10 first; missing rows: {missing}"
assert all(len(runs[n]["per_image"]) == 10 for n in ORDER), "every row must be averaged over the 10 test patches"

In [2]:
def fmt(v):
    return f"{v[0]:.4f} ± {v[1]:.4f}"

print(f"{'Approach':34s}" + "".join(f"{k:>20s}" for k in METRICS) + "   same as Table 3")
print("-" * 134)
n_same = 0
for name in ORDER:
    s = runs[name]["summary"]
    same = matches(s, name) if name in TABLE3 else None
    n_same += bool(same)
    flag = {True: "✓", False: "✗", None: "-"}[same]
    print(f"{name:34s}" + "".join(f"{fmt(s[k]):>20s}" for k in METRICS) + f"   {flag}")
print(f"\n{n_same}/{len(ORDER)} rows identical to Table 3 (all four metrics, mean and std, 4 decimals)")

Approach                                          PSNR                SSIM                  RE                RMSE   same as Table 3
--------------------------------------------------------------------------------------------------------------------------------------
TTSIDC                                36.8247 ± 2.5611     0.9882 ± 0.0076     0.0746 ± 0.0243     0.0262 ± 0.0097   ✓
Oracle TRDC                           40.9487 ± 2.7437     0.9910 ± 0.0051     0.0455 ± 0.0125     0.0160 ± 0.0055   ✓
TRDC                                  37.7190 ± 2.1283     0.9862 ± 0.0084     0.0656 ± 0.0176     0.0231 ± 0.0080   ✓
DU-Net                                36.2222 ± 1.8690     0.9861 ± 0.0088     0.0768 ± 0.0144     0.0270 ± 0.0073   ✓
DC-Net                                33.9575 ± 1.9905     0.9771 ± 0.0116     0.1003 ± 0.0208     0.0353 ± 0.0101   ✓
Physics-Trained                       26.4089 ± 1.7020     0.8490 ± 0.1073     0.2391 ± 0.0506     0.0816 ± 0.0128   ✓
CNN-regularized   

## Write `results/table3.csv` and `results/table3.md`
Bold: best value of each column.

In [3]:
# best row of each column (at full precision, so ties at 4 decimals are broken)
best = {k: (max if k in ("PSNR", "SSIM") else min)(ORDER, key=lambda n: runs[n]["summary"][k][0]) for k in METRICS}

with open(os.path.join(RESULTS_DIR, "table3.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Approach"] + [f"{k}_{stat}" for k in METRICS for stat in ("mean", "std")])
    for name in ORDER:
        w.writerow([name] + [f"{v:.4f}" for k in METRICS for v in runs[name]["summary"][k]])

lines = ["| Approach | PSNR (dB) ↑ | SSIM ↑ | RE ↓ | RMSE ↓ |", "|---|---|---|---|---|"]
for name in ORDER:
    s = runs[name]["summary"]
    cells = [f"**{fmt(s[k])}**" if name == best[k] else fmt(s[k]) for k in METRICS]
    lines.append(f"| {name} | " + " | ".join(cells) + " |")
open(os.path.join(RESULTS_DIR, "table3.md"), "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

| Approach | PSNR (dB) ↑ | SSIM ↑ | RE ↓ | RMSE ↓ |
|---|---|---|---|---|
| TTSIDC | 36.8247 ± 2.5611 | 0.9882 ± 0.0076 | 0.0746 ± 0.0243 | 0.0262 ± 0.0097 |
| Oracle TRDC | 40.9487 ± 2.7437 | 0.9910 ± 0.0051 | 0.0455 ± 0.0125 | 0.0160 ± 0.0055 |
| TRDC | 37.7190 ± 2.1283 | 0.9862 ± 0.0084 | 0.0656 ± 0.0176 | 0.0231 ± 0.0080 |
| DU-Net | 36.2222 ± 1.8690 | 0.9861 ± 0.0088 | 0.0768 ± 0.0144 | 0.0270 ± 0.0073 |
| DC-Net | 33.9575 ± 1.9905 | 0.9771 ± 0.0116 | 0.1003 ± 0.0208 | 0.0353 ± 0.0101 |
| Physics-Trained | 26.4089 ± 1.7020 | 0.8490 ± 0.1073 | 0.2391 ± 0.0506 | 0.0816 ± 0.0128 |
| CNN-regularized | 28.3958 ± 1.7156 | 0.9254 ± 0.0316 | 0.1888 ± 0.0293 | 0.0663 ± 0.0158 |
| U-Net | 32.4771 ± 1.9784 | 0.9724 ± 0.0142 | 0.1201 ± 0.0277 | 0.0421 ± 0.0119 |
| Denoising diffusion (U-Net) | 38.4493 ± 2.7439 | 0.9911 ± 0.0089 | 0.0618 ± 0.0214 | 0.0218 ± 0.0091 |
| Multiscale DiT (Single-shot) | 39.8545 ± 2.6601 | 0.9940 ± 0.0067 | 0.0525 ± 0.0181 | 0.0186 ± 0.0078 |
| Multiscale DiT (Stabi